In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy as scp

def pricing_asian_call(S, K, q, T, r, daily_vol, batch_size=1000, N=1000000, confidence_lvl=0.99, decimal=3):

    # S: Spot price, K: Strike price, q: Dividend yield, T: Maturity (years), r: Risk-free rate, daily_vol: Daily volatility

    sigma = daily_vol*np.sqrt(252) #Annualized volatility

    steps = int(T*252) # Total number of discrete price points in the path
    dt = T/steps   # Time increment per step

    # In order to limit RAM usage, we separate our realizations of the option in batches
    batch_num = N // batch_size #We round to the nearest whole number of batches needed to do N realizations

    Payoffs = []

    for _ in range(batch_num):

        # Path Generation (Geometric Brownian Motion)
        Z=np.random.normal( size=(steps,batch_size) )

        # We calculate the log-returns increments: (r - q - 0.5 * sigma^2)dt + sigma * sqrt(dt) * Z
        increments = (r - q - sigma**2/2)*dt + sigma*np.sqrt(dt) * Z
        
        # We ensures every path starts at the current Spot Price S
        # We then cumulate returns, and apply to Spot S
        Paths = S * np.exp(np.vstack([np.zeros(batch_size), np.cumsum(increments, axis=0)]))

        # We calculate the arithmetic average of the price path for each realization
        Mean = np.mean(Paths, axis=0)

        # We then calculate the payoff for the Call: max(Average_Price - Strike, 0)
        for i in range(len(Mean)):
            Payoffs.append(max(Mean[i]-K,0))

    # Get the average of our realizations and then discount the average payoff back to present value using e^(-rT)
    Call_price = np.mean(Payoffs)*np.exp(-r*T)

    # Calculate the standard deviation of the discounted payoffs to find the error
    sigma_payoff = np.std(Payoffs) * np.exp(-r*T)

    # We define the radius of the Confidence Interval for our price estimate using the Central Limit Theorem.
    Error_margin = scp.stats.norm.ppf(1-(1-confidence_lvl)/2)* ( sigma_payoff / np.sqrt(N) )

    return(
    f"The option price is : {round(Call_price,decimal+1)} ± {round(Error_margin,decimal+1)}, at confidence level {confidence_lvl*100}% \n"
    f"In the following interval :  [{round(Call_price - Error_margin , decimal )} ; {round( Call_price + Error_margin, decimal)}]" )

In [ ]:
#Test with S=100 , K=120 , dividends=2% , T= 1year , risk free rate = 2% and daily vol = 1%
print(pricing_asian_call(100, 120, 0.02, 1, 0.02, 0.01, decimal=5, N=10000000))

The option price is : 0.09569 ± 0.000713, at confidence level 99.0% 
In the following interval :  [0.09498 ; 0.0964]
